In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit_aer
!pip install pylatexenc

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [ ]:
import numpy as np

seed = 1234
np.random.seed(seed)
reps = 1000

x = np.random.randint(2, size = reps)
y = np.random.randint(2, size = reps)

In [ ]:
circuit_list = []

for i in range(reps):

    circuit = QuantumCircuit(2)

    # Bell state preparation
    circuit.h(0)
    circuit.cx(0,1)

    circuit.barrier()

    # Alice and Bob's ops

    if x[i] == 1:
        circuit.ry(np.pi/2,0)
    if y[i] == 0:
        circuit.ry(np.pi/4,1)
    else:
        circuit.ry(-np.pi/4,1)
    circuit.measure_all()

    circuit_list.append(circuit)

In [ ]:
sampler = Sampler(AerSimulator(seed_simulator = seed))

job = sampler.run(circuit_list, shots = 1)
results = job.result()

wins = 0

for i in range(reps):
    bits = results[i].data.meas.get_bitstrings()
    a = int(bits[0][1])
    b = int(bits[0][0])

    if x[i]*y[i] == 0:
        if a == b:
            wins+=1
    else:
        if a != b:
            wins += 1

print("Win percentage:", 100*wins/reps)

##BONUS: EJECUCIÓN EN UN ORDENADOR CUÁNTICO REAL

In [ ]:
mytoken = ""

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_cloud", token=mytoken)

In [ ]:
backend = service.least_busy(simulator=False, operational=True)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

isa_circuit_list = [pm.run(qc) for qc in circuit_list]



In [ ]:
sampler = Sampler(backend)

job = sampler.run(isa_circuit_list, shots = 1)
results = job.result()

wins = 0

for i in range(reps):
    bits = results[i].data.meas.get_bitstrings()
    a = int(bits[0][1])
    b = int(bits[0][0])

    if x[i]*y[i] == 0:
        if a == b:
            wins+=1
    else:
        if a != b:
            wins += 1

print("Win percentage:", 100*wins/reps)